# Chapter 5 — Text Clustering and Topic Modeling
### Practice Notebook

*Source: Hands-On Large Language Models, Jay Alammar & Maarten Grootendorst (O'Reilly)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter05/Chapter%205%20-%20Text%20Clustering%20and%20Topic%20Modeling.ipynb)

---

Text clustering and topic modeling are **unsupervised** techniques — no labels required. The chapter builds a full pipeline from raw text to interpretable topics:

```
Raw text
  → [1] Embed documents         (sentence-transformers, 384-dim)
  → [2] Reduce dimensionality   (UMAP, 384 → 5 dims)
  → [3] Cluster                 (HDBSCAN, 156 clusters)
  → [4] Represent topics        (c-TF-IDF keywords)
  → [5] Fine-tune labels        (KeyBERT / MMR / Flan-T5 / GPT)
```

---

## Table of Contents

- [Part 1: The Text Clustering Pipeline](#part-1-the-text-clustering-pipeline)
  - [Exercise 1.1 — Load the ArXiv Dataset](#exercise-11--load-the-arxiv-dataset)
  - [Exercise 1.2 — Embed Documents](#exercise-12--embed-documents)
  - [Exercise 1.3 — Reduce Dimensionality with UMAP](#exercise-13--reduce-dimensionality-with-umap)
  - [Exercise 1.4 — Cluster with HDBSCAN](#exercise-14--cluster-with-hdbscan)
  - [Exercise 1.5 — Inspect Cluster Contents](#exercise-15--inspect-cluster-contents)
  - [Exercise 1.6 — Visualise Clusters in 2D](#exercise-16--visualise-clusters-in-2d)
- [Part 2: Theory — UMAP Parameters and HDBSCAN vs k-Means](#part-2-theory--umap-parameters-and-hdbscan-vs-k-means)
  - [Exercise 2.1 — Effect of UMAP n_components](#exercise-21--effect-of-umap-n_components)
  - [Exercise 2.2 — HDBSCAN vs k-Means](#exercise-22--hdbscan-vs-k-means)
- [Part 3: BERTopic — Automated Topic Modeling](#part-3-bertopic--automated-topic-modeling)
  - [Exercise 3.1 — Train BERTopic](#exercise-31--train-bertopic)
  - [Exercise 3.2 — Explore Topics](#exercise-32--explore-topics)
  - [Exercise 3.3 — Inspect and Search Topics](#exercise-33--inspect-and-search-topics)
- [Part 4: Visualising Topics](#part-4-visualising-topics)
  - [Exercise 4.1 — Document Map](#exercise-41--document-map)
  - [Exercise 4.2 — Keyword Barchart and Heatmap](#exercise-42--keyword-barchart-and-heatmap)
- [Part 5: Fine-Tuning Topic Representations](#part-5-fine-tuning-topic-representations)
  - [Exercise 5.1 — KeyBERTInspired](#exercise-51--keyBERTInspired)
  - [Exercise 5.2 — Maximal Marginal Relevance](#exercise-52--maximal-marginal-relevance)
  - [Exercise 5.3 — Flan-T5 Text Generation Labels](#exercise-53--flan-t5-text-generation-labels)
  - [Exercise 5.4 — OpenAI GPT Labels (Optional)](#exercise-54--openai-gpt-labels-optional)
- [Part 6: c-TF-IDF From Scratch](#part-6-c-tf-idf-from-scratch)

In [ ]:
# %%capture
# !pip install bertopic datasets openai datamapplot

---
# Part 1: The Text Clustering Pipeline

We build the three-step clustering pipeline manually before using BERTopic to automate it. The dataset is 44,949 ArXiv abstracts from the NLP field (1991–2024).

💡 **GPU recommended** for the embedding step.

## Exercise 1.1 — Load the ArXiv Dataset

**Task:**
1. Load `"maartengr/arxiv_nlp"` from HuggingFace datasets, take the `"train"` split
2. Extract `abstracts` and `titles` as plain Python lists
3. Print how many abstracts there are and the first abstract (truncated to 300 chars)

In [ ]:
from datasets import load_dataset

# YOUR CODE HERE
# dataset = load_dataset("maartengr/arxiv_nlp")["train"]
# abstracts = list(dataset["Abstracts"])
# titles    = list(dataset["Titles"])

# print(f"Number of abstracts: {len(abstracts)}")
# print(f"\nFirst abstract (first 300 chars):")
# print(abstracts[0][:300])

## Exercise 1.2 — Embed Documents

We convert every abstract into a 384-dimensional dense vector using a sentence embedding model. We use `thenlper/gte-small` — a small but strong model optimised for semantic similarity.

**Task:**
1. Load `SentenceTransformer('thenlper/gte-small')`
2. Encode all abstracts with `show_progress_bar=True`
3. Print the shape of the resulting embeddings matrix

**Expected shape:** `(44949, 384)`

In [ ]:
from sentence_transformers import SentenceTransformer

# YOUR CODE HERE
# embedding_model = SentenceTransformer('thenlper/gte-small')
# embeddings = embedding_model.encode(abstracts, show_progress_bar=True)
# print(f"Embeddings shape: {embeddings.shape}")
# Expected: (44949, 384)

## Exercise 1.3 — Reduce Dimensionality with UMAP

Clustering in 384 dimensions is problematic — distances become meaningless in very high-dimensional spaces (the **curse of dimensionality**). UMAP compresses the embeddings to a much lower dimension while preserving global structure.

**Key parameters:**
| Parameter | Value | Why |
|---|---|---|
| `n_components` | 5 | Target dimensions (5–10 preserves global structure well) |
| `min_dist` | 0.0 | How tightly to pack points; 0 gives tightest clusters |
| `metric` | `'cosine'` | Cosine distance works better than Euclidean in high dimensions |
| `random_state` | 42 | Reproducibility (disables parallelism as a side-effect) |

**Task:** Create and fit a UMAP model with the parameters above. Print the shape of `reduced_embeddings`.

**Expected shape:** `(44949, 5)`

In [ ]:
from umap import UMAP

# YOUR CODE HERE
# umap_model = UMAP(
#     n_components=5, min_dist=0.0, metric='cosine', random_state=42
# )
# reduced_embeddings = umap_model.fit_transform(embeddings)
# print(f"Reduced embeddings shape: {reduced_embeddings.shape}")
# Expected: (44949, 5)

## Exercise 1.4 — Cluster with HDBSCAN

HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise) finds clusters without requiring you to specify how many there are. It also identifies **outliers** — data points that don't belong to any cluster — and assigns them label `-1`.

This differs from k-means, which forces every point into a cluster and requires you to pre-specify `k`.

**Task:**
1. Fit HDBSCAN with `min_cluster_size=50`, `metric='euclidean'`, `cluster_selection_method='eom'`
2. Extract the cluster labels
3. Print the total number of unique clusters (excluding `-1`)
4. Print how many abstracts are outliers (label == -1)

**Expected:** ~156 clusters, ~1420 outliers

In [ ]:
from hdbscan import HDBSCAN
import numpy as np

# YOUR CODE HERE
# hdbscan_model = HDBSCAN(
#     min_cluster_size=50, metric='euclidean', cluster_selection_method='eom'
# ).fit(reduced_embeddings)
# clusters = hdbscan_model.labels_

# n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
# n_outliers = (clusters == -1).sum()
# print(f"Number of clusters: {n_clusters}")
# print(f"Number of outliers (label=-1): {n_outliers}")

## Exercise 1.5 — Inspect Cluster Contents

Clustering is unsupervised — we have no predefined labels. To understand what a cluster contains, we manually read a few documents from it.

**Task:**
1. Print the first 3 abstracts from cluster `0` (first 300 chars each)
2. Print the first 3 abstracts from cluster `1`
3. Based on the content, guess what topic each cluster represents

**Hint:** `np.where(clusters == cluster_id)[0][:3]` gives the indices of the first 3 documents in a cluster.

In [ ]:
# YOUR CODE HERE
# for cluster_id in [0, 1]:
#     print(f"\n{'='*60}")
#     print(f"CLUSTER {cluster_id} — first 3 abstracts:")
#     print('='*60)
#     for idx in np.where(clusters == cluster_id)[0][:3]:
#         print(abstracts[idx][:300] + "...\n")

**What is cluster 0 about?** *(write your guess here)*

**What is cluster 1 about?** *(write your guess here)*

## Exercise 1.6 — Visualise Clusters in 2D

We can only visualise in 2D, so we create a second UMAP reduction with `n_components=2`. This is **for visualisation only** — the clustering was done on the 5-dimensional reduction.

**Task:**
1. Fit a new UMAP with `n_components=2` (same other params) on the original 384-dim `embeddings`
2. Create a DataFrame with columns `x`, `y`, `title`, `cluster`
3. Separate into `clusters_df` (non-outliers) and `outliers_df` (cluster == "-1")
4. Plot with matplotlib: outliers in gray (alpha=0.05), clusters coloured by cluster ID

**Important note:** The 2D plot is lossy — it's for exploration, not ground truth. Clusters that look close in 2D may actually be far apart in the higher-dimensional space.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# YOUR CODE HERE
# Step 1: 2D UMAP for visualisation
# reduced_2d = UMAP(n_components=2, min_dist=0.0, metric='cosine', random_state=42).fit_transform(embeddings)

# Step 2: DataFrame
# df = pd.DataFrame(reduced_2d, columns=["x", "y"])
# df["title"]   = titles
# df["cluster"] = [str(c) for c in clusters]

# Step 3: Split
# clusters_df = df.loc[df.cluster != "-1", :]
# outliers_df = df.loc[df.cluster == "-1", :]

# Step 4: Plot
# plt.figure(figsize=(12, 8))
# plt.scatter(outliers_df.x, outliers_df.y, alpha=0.05, s=2, c="gray")
# plt.scatter(
#     clusters_df.x, clusters_df.y,
#     c=clusters_df.cluster.astype(int), alpha=0.6, s=2, cmap="tab20b"
# )
# plt.axis("off")
# plt.title("ArXiv NLP abstracts — 2D cluster map")
# plt.show()

---
# Part 2: Theory — UMAP Parameters and HDBSCAN vs k-Means

Understanding the *why* behind parameter choices helps you adapt the pipeline to new datasets.

## Exercise 2.1 — Effect of UMAP `n_components`

`n_components` controls how much we compress the embeddings. Too low → too much information lost. Too high → curse of dimensionality still applies for clustering.

**Task:** Run UMAP with three different `n_components` values, then HDBSCAN on each, and compare the number of clusters found.

```python
n_components_values = [2, 5, 15]
```

Print: `n_components → number of clusters found → number of outliers`

**Observe:** More components generally preserves more structure → more fine-grained clusters. Fewer components → fewer, broader clusters.

In [ ]:
# YOUR CODE HERE
# Note: This will take a few minutes — UMAP is O(n log n)
# n_components_values = [2, 5, 15]

# for nc in n_components_values:
#     reduced = UMAP(n_components=nc, min_dist=0.0, metric='cosine', random_state=42).fit_transform(embeddings)
#     labels = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom').fit(reduced).labels_
#     n_clust = len(set(labels)) - (1 if -1 in labels else 0)
#     n_out   = (labels == -1).sum()
#     print(f"n_components={nc:2d} → {n_clust:3d} clusters, {n_out:4d} outliers")

## Exercise 2.2 — HDBSCAN vs k-Means

k-Means is a centroid-based algorithm: it assigns **every** point to a cluster and you must pre-specify `k`. HDBSCAN is density-based: it discovers `k` automatically and can leave outliers unassigned.

**Task:** Run k-Means with `k=156` (the number BERTopic found) on the 5-dim reduced embeddings and compare:
1. Does k-Means produce any outliers?
2. Are the cluster sizes similar between HDBSCAN and k-Means?

Print the 10 largest cluster sizes for each method.

In [ ]:
from sklearn.cluster import KMeans
from collections import Counter

# YOUR CODE HERE
# kmeans = KMeans(n_clusters=156, random_state=42, n_init=10)
# kmeans_labels = kmeans.fit_predict(reduced_embeddings)

# print("HDBSCAN — 10 largest clusters:")
# hdbscan_counts = Counter(clusters)
# hdbscan_counts.pop(-1, None)  # remove outlier class
# for label, count in hdbscan_counts.most_common(10):
#     print(f"  Cluster {label:3d}: {count} docs")

# print("\nk-Means — 10 largest clusters:")
# for label, count in Counter(kmeans_labels).most_common(10):
#     print(f"  Cluster {label:3d}: {count} docs")

# print(f"\nk-Means outliers: {(kmeans_labels == -1).sum()}  (always 0 — k-means forces all points into clusters)")

---
# Part 3: BERTopic — Automated Topic Modeling

**BERTopic** automates the entire pipeline and adds topic representation via **c-TF-IDF** — a class-based variant of TF-IDF that ranks words by how much they characterise a cluster compared to all other clusters.

BERTopic is modular: you plug in your own embedding model, UMAP, and HDBSCAN, and it handles the rest.

## Exercise 3.1 — Train BERTopic

**Task:** Train BERTopic by passing in the pre-built components (embedding model, UMAP model, HDBSCAN model) and the pre-computed embeddings. Set `verbose=True`.

**Hint:** Pass `embeddings` directly to `.fit()` to skip re-encoding:
```python
topic_model = BERTopic(...).fit(abstracts, embeddings)
```

In [ ]:
from bertopic import BERTopic

# YOUR CODE HERE
# topic_model = BERTopic(
#     embedding_model=embedding_model,
#     umap_model=umap_model,
#     hdbscan_model=hdbscan_model,
#     verbose=True
# ).fit(abstracts, embeddings)

# print("BERTopic trained successfully.")

## Exercise 3.2 — Explore Topics

**Task:**
1. Call `topic_model.get_topic_info()` to get the full topic table
2. Print it — it has columns: `Topic`, `Count`, `Name`, `Representation`
3. Print the total number of topics (excluding topic -1)
4. Print the topic with the **most** documents

**Note:** Topic `-1` = outliers. Its representation `"1_the_of_and_to"` is meaningless stop words — HDBSCAN outliers have no real theme.

In [ ]:
# YOUR CODE HERE
# topic_info = topic_model.get_topic_info()
# print(topic_info)

# n_real_topics = len(topic_info[topic_info.Topic != -1])
# print(f"\nNumber of real topics: {n_real_topics}")

# biggest = topic_info[topic_info.Topic != -1].iloc[0]  # already sorted by Count
# print(f"Largest topic: Topic {biggest.Topic} ({biggest.Count} docs) — {biggest.Name}")

## Exercise 3.3 — Inspect and Search Topics

**Task:**
1. Use `topic_model.get_topic(0)` to get the top keywords and scores for Topic 0. Print them formatted.
2. Use `topic_model.find_topics("topic modeling")` to find which topic covers topic modeling research. Print the topic IDs and similarity scores.
3. Inspect that topic with `get_topic(topic_id)` — confirm it's about LDA and topic modeling.

**Expected:** `find_topics("topic modeling")` should return topic 22 with high similarity (~0.95).

In [ ]:
# YOUR CODE HERE

# 1. Inspect Topic 0
# print("Topic 0 keywords:")
# for word, score in topic_model.get_topic(0):
#     print(f"  {word:<20}: {score:.4f}")

# 2. Search for topic modeling
# topic_ids, scores = topic_model.find_topics("topic modeling")
# print(f"\nTop topics matching 'topic modeling':")
# for tid, score in zip(topic_ids[:3], scores[:3]):
#     name = topic_model.get_topic_info(tid)["Name"].values[0]
#     print(f"  Topic {tid}: similarity={score:.4f} | name={name}")

# 3. Inspect the top match
# best_topic = topic_ids[0]
# print(f"\nKeywords for topic {best_topic}:")
# for word, score in topic_model.get_topic(best_topic):
#     print(f"  {word:<20}: {score:.4f}")

---
# Part 4: Visualising Topics

BERTopic has built-in interactive visualisations. These are the most useful tools for exploring what your topics contain.

## Exercise 4.1 — Document Map

**Task:** Call `topic_model.visualize_documents()` passing the paper titles and the 2D reduced embeddings. Use `width=1200` and `hide_annotations=True`.

Then update the font size with `fig.update_layout(font=dict(size=10))` and display.

*This produces an interactive plot — hover over points to see paper titles and topics.*

In [ ]:
# YOUR CODE HERE
# fig = topic_model.visualize_documents(
#     titles,
#     reduced_embeddings=reduced_2d,
#     width=1200,
#     hide_annotations=True
# )
# fig.update_layout(font=dict(size=10))
# fig.show()

## Exercise 4.2 — Keyword Barchart and Heatmap

**Task:** Generate three visualisations:
1. `topic_model.visualize_barchart()` — ranked keywords per topic
2. `topic_model.visualize_heatmap(n_clusters=30)` — similarity between topics
3. `topic_model.visualize_hierarchy()` — hierarchical tree of topics

For each, call `.show()` to display.

In [ ]:
# YOUR CODE HERE
# topic_model.visualize_barchart().show()
# topic_model.visualize_heatmap(n_clusters=30).show()
# topic_model.visualize_hierarchy().show()

---
# Part 5: Fine-Tuning Topic Representations

c-TF-IDF gives us keyword lists, but the keywords can be redundant ("summary", "summaries", "summarization" all in the same topic) or missing context. BERTopic's **representation models** rerank or replace these keywords using more powerful techniques.

The pipeline is: `c-TF-IDF → [representation model] → improved topic labels`

Crucially, representation models only run **once per topic** (not once per document), making them efficient even for large corpora.

In [ ]:
from copy import deepcopy

# Save original topic representations for comparison
original_topics = deepcopy(topic_model.topic_representations_)


def topic_differences(model, original_topics, nr_topics=5):
    """Show original vs updated topic keywords side by side."""
    import pandas as pd
    rows = []
    for topic in range(nr_topics):
        og_words  = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        rows.append({"Topic": topic, "Original (c-TF-IDF)": og_words, "Updated": new_words})
    return pd.DataFrame(rows)

## Exercise 5.1 — KeyBERTInspired

KeyBERTInspired uses cosine similarity between word embeddings and the average document embedding of each topic. It selects words that are semantically closest to the topic's centroid — this removes stop words naturally (they have generic embeddings far from the topic centroid).

**Task:**
1. Create a `KeyBERTInspired()` representation model
2. Call `topic_model.update_topics(abstracts, representation_model=representation_model)`
3. Display the differences using `topic_differences()`

In [ ]:
from bertopic.representation import KeyBERTInspired

# YOUR CODE HERE
# representation_model = KeyBERTInspired()
# topic_model.update_topics(abstracts, representation_model=representation_model)
# topic_differences(topic_model, original_topics)

**Observation:** Do you see stop words being removed? Are the updated keywords more informative than the originals? *(write here)*

## Exercise 5.2 — Maximal Marginal Relevance

KeyBERTInspired can produce redundant keywords ("summary", "summaries", "summarization"). **MMR** (Maximal Marginal Relevance) addresses this by iteratively selecting the next keyword that is both relevant to the topic AND maximally different from already-selected keywords.

The `diversity` parameter (0–1) controls the trade-off: 0 = pure relevance, 1 = pure diversity.

**Task:**
1. Try MMR with `diversity=0.2` (low diversity — slight deduplication)
2. Try MMR with `diversity=0.7` (high diversity — very different keywords)
3. Compare both against the original c-TF-IDF keywords

In [ ]:
from bertopic.representation import MaximalMarginalRelevance

# YOUR CODE HERE
# for diversity in [0.2, 0.7]:
#     representation_model = MaximalMarginalRelevance(diversity=diversity)
#     topic_model.update_topics(abstracts, representation_model=representation_model)
#     print(f"\nMMR diversity={diversity}:")
#     print(topic_differences(topic_model, original_topics).to_string(index=False))

**Observation:** At high diversity, keywords become more different from each other. Does high diversity produce more or less informative labels for this dataset? *(write here)*

## Exercise 5.3 — Flan-T5 Text Generation Labels

Instead of reranking keywords, we can ask a **generative model** to create a single descriptive label per topic. The model receives the c-TF-IDF keywords and a few representative documents, and generates a short label.

This runs **once per topic** (not once per document), so even for 156 topics it is fast.

**Task:**
1. Create a Flan-T5 pipeline (`google/flan-t5-small`, `text2text-generation`)
2. Create a `TextGeneration` representation model with the prompt below
3. Update topics and show the differences

**Expected:** Labels like "Speech-to-description", "Attention-based neural machine translation", "Summarization"

In [ ]:
from transformers import pipeline as hf_pipeline
from bertopic.representation import TextGeneration

prompt = """I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS].

Based on the documents and keywords, what is this topic about?"""

# YOUR CODE HERE
# generator = hf_pipeline("text2text-generation", model="google/flan-t5-small")
# representation_model = TextGeneration(
#     generator, prompt=prompt, doc_length=50, tokenizer="whitespace"
# )
# topic_model.update_topics(abstracts, representation_model=representation_model)
# topic_differences(topic_model, original_topics)

## Exercise 5.4 — OpenAI GPT Labels (Optional)

**⚠️ Requires an OpenAI API key.** GPT-3.5 produces more informative and fluent labels than Flan-T5 but costs money (~0.03 cents per topic at gpt-3.5-turbo pricing).

**Task:** Implement the OpenAI representation model using the prompt template below. Run on the first 5 topics only to minimise cost.

**Expected labels:** "Neural Machine Translation Enhancements", "Document Summarization Techniques", "Advancements in Aspect-Based Sentiment Analysis"

In [ ]:
# YOUR CODE HERE — skip if you want to save API credits
# import openai
# from bertopic.representation import OpenAI as BERTopicOpenAI

# client = openai.OpenAI(api_key="YOUR_KEY_HERE")

# gpt_prompt = """
# I have a topic that contains the following documents:
# [DOCUMENTS]
#
# The topic is described by the following keywords: [KEYWORDS]
#
# Based on the information above, extract a short topic label in the following format:
# topic: <short topic label>
# """

# representation_model = BERTopicOpenAI(
#     client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=gpt_prompt
# )
# topic_model.update_topics(abstracts, representation_model=representation_model)
# topic_differences(topic_model, original_topics)

---
# Part 6: c-TF-IDF From Scratch

BERTopic's secret sauce is **c-TF-IDF** (class-based TF-IDF). This part implements it manually on a tiny corpus so you see exactly what the formula is computing.

**Standard TF-IDF** weights words by how often they appear in a *document* vs how often they appear across all *documents*.

**c-TF-IDF** does the same but at the *cluster* level: it weights words by how often they appear in a *cluster* vs how often they appear across all *clusters*.

$$c\text{-TF-IDF}_t = \text{tf}_c(t) \times \log\left(\frac{A}{f_t} + 1\right)$$

Where:
- $\text{tf}_c(t)$ = frequency of word $t$ in cluster $c$
- $A$ = average number of words per cluster
- $f_t$ = total frequency of word $t$ across all clusters

## Exercise 6.1 — Implement c-TF-IDF on a Tiny Corpus

We use a 6-document, 2-cluster toy corpus:

```
Cluster 0 (animals): 
  "my cat is cute"
  "the dog is friendly"
  "my cat and dog play"

Cluster 1 (food): 
  "pasta and pizza are delicious"
  "pizza is my favourite food"
  "cooking pasta is easy"
```

**Task:** Implement `ctf_idf(corpus_by_cluster)` that:
1. Counts word frequencies per cluster (class-based TF)
2. Counts total word frequencies across all clusters (for IDF)
3. Computes $A$ = average words per cluster
4. Computes c-TF-IDF for each word in each cluster
5. Returns the top 3 words per cluster by c-TF-IDF score

**Expected top words:**
- Cluster 0: `cat`, `dog`, `my` (animal-specific)
- Cluster 1: `pasta`, `pizza`, `cooking` (food-specific)

*Note: words like "is" and "and" appear in both clusters → low c-TF-IDF → filtered out*

In [ ]:
import math
from collections import Counter

corpus_by_cluster = {
    0: ["my cat is cute", "the dog is friendly", "my cat and dog play"],
    1: ["pasta and pizza are delicious", "pizza is my favourite food", "cooking pasta is easy"],
}


def ctf_idf(corpus_by_cluster: dict, top_n: int = 3) -> dict:
    """
    Compute c-TF-IDF scores and return top_n words per cluster.
    corpus_by_cluster: {cluster_id: [list of document strings]}
    Returns: {cluster_id: [(word, score), ...]}
    """
    # YOUR CODE HERE

    # Step 1: Count word frequencies per cluster
    # cluster_word_counts = {}  # {cluster_id: Counter}
    # for cluster_id, docs in corpus_by_cluster.items():
    #     all_words = " ".join(docs).split()
    #     cluster_word_counts[cluster_id] = Counter(all_words)

    # Step 2: Count total frequency of each word across ALL clusters
    # total_word_freq = Counter()
    # for counts in cluster_word_counts.values():
    #     total_word_freq.update(counts)

    # Step 3: Compute A = average number of words per cluster
    # total_words_per_cluster = [sum(c.values()) for c in cluster_word_counts.values()]
    # A = sum(total_words_per_cluster) / len(total_words_per_cluster)

    # Step 4: Compute c-TF-IDF for each word in each cluster
    # result = {}
    # for cluster_id, word_counts in cluster_word_counts.items():
    #     scores = {}
    #     for word, tf in word_counts.items():
    #         idf = math.log(A / total_word_freq[word] + 1)
    #         scores[word] = tf * idf
    #     result[cluster_id] = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    # return result
    pass


# result = ctf_idf(corpus_by_cluster)
# for cluster_id, top_words in result.items():
#     print(f"Cluster {cluster_id}: {top_words}")

---
## Chapter 5 Summary

| Concept | What you built | Key insight |
|---|---|---|
| Text clustering pipeline | Embed → UMAP → HDBSCAN | 3 independent, swappable steps |
| UMAP parameters | n_components experiment | More components → more clusters; balance against curse of dimensionality |
| HDBSCAN vs k-Means | Comparison of cluster counts and outlier handling | HDBSCAN discovers k and handles outliers; k-Means forces all points into clusters |
| BERTopic | Full pipeline + topic inspection | Automates clustering + adds c-TF-IDF keywords |
| Visualisation | Document map, barchart, heatmap, hierarchy | 2D is lossy — always inspect cluster contents too |
| KeyBERTInspired | Semantic reranking of keywords | Removes stop words, keeps semantically relevant terms |
| MMR | Diversity-aware keyword selection | Reduces redundancy ("summary", "summaries", "summarization" → one of each) |
| Flan-T5 labels | Text generation for topic labelling | One model call per topic, not per document |
| c-TF-IDF from scratch | Manual implementation on toy corpus | Words distinctive to a cluster rank high; cross-cluster words rank low |